In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
import math

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr')

df = load_dataset('크루즈입항')

In [ ]:
# ============================================================
# 1. 라이브러리 불러오기
# ============================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go


# ============================================================
# 2. 데이터 확인 및 전처리
# ============================================================

if df is None:
    raise ValueError("df가 None입니다. 데이터 로딩을 먼저 실행해주세요.")

if not isinstance(df, pd.DataFrame):
    raise TypeError("df가 pandas DataFrame이 아닙니다.")

frequency_source = df.copy()
frequency_source.columns = frequency_source.columns.str.strip()

required_columns = ["입출", "선박용도", "입항일시"]

missing_columns = [
    col for col in required_columns
    if col not in frequency_source.columns
]

if missing_columns:
    raise ValueError(f"필요한 컬럼이 없습니다: {missing_columns}")


# ============================================================
# 3. 크루즈 입항 데이터 추출
# ============================================================

monthly_data = frequency_source[
    (frequency_source["입출"] == "입항") &
    (frequency_source["선박용도"] == "크루즈선")
].copy()

monthly_data["입항일시"] = pd.to_datetime(
    monthly_data["입항일시"],
    errors="coerce"
)

# 분석 기간: 2013.01.01 ~ 2026.10.16
monthly_data = monthly_data[
    (monthly_data["입항일시"] >= "2013-01-01") &
    (monthly_data["입항일시"] < "2026-10-17")
].copy()

if monthly_data.empty:
    raise ValueError("분석 기간에 해당하는 크루즈 입항 데이터가 없습니다.")


# 연도와 월 추출
monthly_data["연도"] = monthly_data["입항일시"].dt.year
monthly_data["월"] = monthly_data["입항일시"].dt.month


# ============================================================
# 4. 연도별 월별 입항 건수 집계
# ============================================================

year_range = range(2013, 2027)
month_range = range(1, 13)

# 연도별·월별 입항 건수
monthly_counts = (
    monthly_data
    .groupby(["연도", "월"])
    .size()
)

# 전체 연도와 월 조합 생성
# 기록이 없는 연월은 0건으로 처리
full_index = pd.MultiIndex.from_product(
    [year_range, month_range],
    names=["연도", "월"]
)

monthly_counts = monthly_counts.reindex(
    full_index,
    fill_value=0
)


# ============================================================
# 5. 월별 평균 입항 건수 계산
# ============================================================

monthly_average = []

for month in month_range:

    # 1~10월: 2013~2026년 (14개 연도)
    # 11~12월: 2013~2025년 (13개 연도)
    if month <= 10:
        valid_years = range(2013, 2027)
    else:
        valid_years = range(2013, 2026)

    # 해당 월의 연도별 입항 건수 추출
    yearly_counts = (
        monthly_counts
        .xs(month, level="월")
        .reindex(valid_years, fill_value=0)
    )

    # 연도별 평균 계산
    average_count = yearly_counts.mean()

    monthly_average.append({
        "월": month,
        "평균입항건수": average_count
    })


# 그래프용 DataFrame 생성
chart_data = pd.DataFrame(monthly_average)

chart_data["월표시"] = (
    chart_data["월"].astype(str) + "월"
)


# ============================================================
# 6. 도넛그래프용 비중 계산 및 색상 설정
# ============================================================

# 각 월의 평균 입항건수를 12개월 평균값 합계로 나누어 비중 계산
average_total = chart_data["평균입항건수"].sum()

if average_total <= 0:
    raise ValueError("월별 평균 입항건수 합계가 0 이하입니다.")

chart_data["비중"] = (
    chart_data["평균입항건수"] / average_total * 100
)

# 대시보드 참고 이미지와 어울리는 12색 팔레트
month_colors = [
    "#1769D2",  # 1월
    "#4A9FE8",  # 2월
    "#20B8C8",  # 3월
    "#27B991",  # 4월
    "#8BCB4D",  # 5월
    "#E5C447",  # 6월
    "#F2A23A",  # 7월
    "#EF7058",  # 8월
    "#E65B83",  # 9월
    "#D77DB8",  # 10월
    "#A58BDD",  # 11월
    "#7D91E5"   # 12월
]


# ============================================================
# 7. 도넛그래프 생성
# ============================================================

fig_monthly_average = go.Figure()

fig_monthly_average.add_trace(
    go.Pie(
        labels=chart_data["월표시"],
        values=chart_data["평균입항건수"],

        # 도넛 형태
        hole=0.58,
        sort=False,
        direction="clockwise",
        rotation=90,

        # 도넛은 왼쪽, 범례는 오른쪽에 별도 배치
        domain=dict(
            x=[0.04, 0.57],
            y=[0.12, 0.86]
        ),

        marker=dict(
            colors=month_colors,
            line=dict(
                color="white",
                width=2
            )
        ),

        # 조각 내부에는 텍스트를 넣지 않고 오른쪽 범례에 월과 비중 표시
        textinfo="none",
        showlegend=False,

        customdata=chart_data[
            ["평균입항건수", "비중"]
        ].to_numpy(),

        hovertemplate=(
            "<b>%{label}</b>"
            "<br>월별 평균 입항건수: %{customdata[0]:.1f}건"
            "<br>월별 평균값 합계 대비 비중: %{customdata[1]:.1f}%"
            "<extra></extra>"
        )
    )
)


# ============================================================
# 8. 그래프 레이아웃 및 중앙 텍스트
# ============================================================

fig_monthly_average.update_layout(

    title=dict(
        text=(
            "<b>부산항 크루즈 월별 평균 입항건수</b>"
            "<br>"
            "<span style='font-size:23px;color:#657487'>"
            "월별 입항 집중 시기 분석"
            "</span>"
        ),
        x=0.5,
        y=0.96,
        xanchor="center",
        yanchor="top",
        font=dict(
            size=30,
            color="#243247"
        )
    ),

    template="plotly_white",
    width=1650,
    height=900,

    paper_bgcolor="white",
    plot_bgcolor="white",

    font=dict(
        family="Malgun Gothic, Arial, sans-serif",
        size=18,
        color="#243247"
    ),

    margin=dict(
        l=75,
        r=65,
        t=155,
        b=145
    ),

    showlegend=False
)

# 도넛 중앙에는 전체 입항건수와 혼동하지 않도록 지표 이름만 표시
fig_monthly_average.add_annotation(
    x=0.305,
    y=0.49,
    xref="paper",
    yref="paper",
    text="<b>월별 평균</b><br><b>입항건수</b>",
    showarrow=False,
    align="center",
    font=dict(
        family="Malgun Gothic, Arial, sans-serif",
        size=21,
        color="#173455"
    )
)


# ============================================================
# 9. 오른쪽 2열 범례 생성
# ============================================================

# 1~6월은 왼쪽 열, 7~12월은 오른쪽 열
legend_columns = [
    list(range(0, 6)),
    list(range(6, 12))
]

legend_x_positions = [0.63, 0.82]
legend_y_positions = np.linspace(0.75, 0.25, 6)

for column_number, month_indices in enumerate(legend_columns):

    for row_number, index in enumerate(month_indices):

        month_label = chart_data.loc[index, "월표시"]
        percentage = chart_data.loc[index, "비중"]
        color = month_colors[index]

        fig_monthly_average.add_annotation(
            x=legend_x_positions[column_number],
            y=legend_y_positions[row_number],
            xref="paper",
            yref="paper",
            text=(
                f"<span style='color:{color}'>●</span> "
                f"<b>{month_label}</b>　{percentage:.1f}%"
            ),
            showarrow=False,
            xanchor="left",
            yanchor="middle",
            align="left",
            font=dict(
                family="Malgun Gothic, Arial, sans-serif",
                size=18,
                color="#243247"
            )
        )


# ============================================================
# 10. 그래프 하단 주석
# ============================================================

fig_monthly_average.add_annotation(
    x=0,
    y=-0.14,
    xref="paper",
    yref="paper",
    text=(
        "※ 분석 기간: 2013.01.01~2026.10.16"
        "<br>※ 1~10월은 14개 연도, 11~12월은 13개 연도 평균"
        "<br>※ 기록이 없는 연월은 0건으로 처리"
        "<br>※ 범례 비중은 12개월 평균 입항건수 합계 대비 비율"
    ),
    showarrow=False,
    xanchor="left",
    yanchor="top",
    align="left",
    font=dict(
        size=17,
        color="#657487"
    )
)


# ============================================================
# 11. 그래프 출력 및 이미지 저장 옵션
# ============================================================

fig_monthly_average.show(
    config={
        "displaylogo": False,
        "toImageButtonOptions": {
            "format": "png",
            "filename": "PPT_부산항_크루즈_월별_평균_입항건수_도넛그래프",
            "width": 1650,
            "height": 900,
            "scale": 3
        }
    }
)
